# Model 2: Standard Pre-trained BART Baseline

In [1]:
pip install transformers datasets evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d297443831d41f40f964d9235f48fd5bc1c12704273278e9ef3dd159fb9f9d45
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
from datasets import load_dataset
billsum = load_dataset("FiscalNote/billsum")
train_dataset = billsum["train"]
test_dataset = billsum["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

data/ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

In [3]:
#billsum = billsum.train_test_split(test_size=0.2)
sample_train = billsum['train'][0]

print("Keys in the dataset:", sample_train.keys())
print("\nSample Text (First 100 chars):", sample_train['text'][:100])
print("\nSample Summary:", sample_train['summary'])

Keys in the dataset: dict_keys(['text', 'summary', 'title'])

Sample Text (First 100 chars): SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES 
              TO NONPROFIT OR

Sample Summary: Shields a business entity from civil liability relating to any injury or death occurring at a facility of that entity in connection with a use of such facility by a nonprofit organization if: (1) the use occurs outside the scope of business of the business entity; (2) such injury or death occurs during a period that such facility is used by such organization; and (3) the business entity authorized the use of such facility by the organization. 
Makes this Act inapplicable to an injury or death that results from an act or omission of a business entity that constitutes gross negligence or intentional misconduct, including misconduct that: (1) constitutes a hate crime or a crime of violence or act of international terrorism for which the defendant has been convicted in any court; or

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")
def preprocess_function(examples):
    inputs = [doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")

    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/18949 [00:00<?, ? examples/s]

Map:   0%|          | 0/3269 [00:00<?, ? examples/s]

In [6]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-base")

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [7]:
import evaluate
import numpy as np
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

In [8]:
training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_billsum_model",
    eval_strategy="epoch",
    learning_rate=0.00002,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=8,
    predict_with_generate=True,
    fp16=True, #this is False for running on CPU, True for when on GPU
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    #train_dataset=tokenized_billsum["train"],
    #eval_dataset=tokenized_billsum["test"],
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    #tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,1.928516,1.616474,0.248600,0.201200,0.241000,0.242400,21.000000
2,1.734375,1.536099,0.250500,0.204500,0.243000,0.244400,21.000000
3,1.608081,1.503482,0.251700,0.205800,0.244200,0.245500,21.000000
4,1.545366,1.475748,0.251000,0.205800,0.243600,0.245000,21.000000
5,1.510659,1.459877,0.252500,0.207600,0.245200,0.246500,21.000000
6,1.468353,1.451086,0.251900,0.207300,0.244700,0.246100,20.999100
7,1.437952,1.447809,0.251800,0.207700,0.244700,0.246000,20.999100
8,1.423444,1.444668,0.251900,0.207300,0.244800,0.246100,20.999100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9480, training_loss=1.5966254061284448, metrics={'train_runtime': 1109.7208, 'train_samples_per_second': 136.604, 'train_steps_per_second': 8.543, 'total_flos': 9.243116917751808e+16, 'train_loss': 1.5966254061284448, 'epoch': 8.0})

In [9]:
test = billsum["test"][0]["text"]
inputs = tokenizer(test, return_tensors="pt", max_length=1024, truncation=True).input_ids.to(model.device)
summary = model.generate(inputs, max_new_tokens=150, num_beams=4, early_stopping=True)
print("BART Summary:")
print(tokenizer.decode(summary[0], skip_special_tokens=True)) #Printing the summary
print("")
print("Original Human made summary:")
print(billsum["test"][0]["summary"])


BART Summary:
Amends the Water Resources Development Act of 1992 to authorize the Secretary of the Interior to carry out projects for the elimination or control of combined sewer overflows in:  (1) Jackson County, Mississippi; (2) Manchester, New Hampshire; (3) Paterson, Passaic County, and Passaic Valley, New Jersey; (4) the North Hudson Sewerage Authority; and (5) the city of North Hudson. 
Directs the Secretary to establish a grant program to provide technical assistance and technical assistance to States for the construction and operation of alternative water supply projects in Mississippi and New Hampshire.

Original Human made summary:
Amends the Water Resources Development Act of 1999 to: (1) authorize appropriations for FY 1999 through 2009 for implementation of a long-term resource monitoring program with respect to the Upper Mississippi River Environmental Management Program (currently, such funding is designated for a program for the planning, construction, and evaluation of

In [10]:
#Save the model, configuration, and tokenizer to a specific folder
save_directory = "./saved_model_2"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Model and tokenizer saved to {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./saved_model_2


In [11]:
import torch
model_to_save = trainer.model
torch.save(model_to_save.state_dict(), "model2_weights.pt")
from google.colab import files #Don't neeed this if running the file locally, this was since Im on google colab
files.download("model2_weights.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

References:

(Because this is a pre-trained model serving as a baseline, most of this code is based on the below Hugging Face Docs tutorial)

https://huggingface.co/docs/transformers/tasks/summarization

https://huggingface.co/docs/transformers/v5.6.2/en/model_doc/auto#transformers.AutoModelForSeq2SeqLM

https://huggingface.co/datasets/FiscalNote/billsum

https://huggingface.co/docs/transformers/main_classes/model

https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html